# AgroVision MobileNetV2 baseline (Google Colab)

This notebook trains a measured baseline using a new ImageNet-pretrained MobileNetV2. The later EfficientNetV2B0 model is out of scope here. The existing AgroVision application model is neither loaded nor changed.

The seed-42 CSV is the sole source of partition assignments. Its metadata is checked up front for counts and leakage; test images are not decoded or used for training or checkpoint selection. Training and validation alone are used for fitting and model selection. The test image data is used only in the final evaluation section, after the best validation checkpoint has been saved. Do not use test results to make model or hyperparameter decisions.

## Before you run

1. In Colab, choose Runtime → Change runtime type → GPU, then run from top to bottom.
2. Put the files listed below in Google Drive. The ZIP is extracted into temporary Colab storage; the original ZIP is read only.
3. Review the configurable paths and hyperparameters. This notebook does not install packages or start training on your local computer.

The local ZIP's exact Kaggle origin and license were not verified in the Phase 2 audit. This limitation is recorded in the report; confirm source, version, and license before publication.

## Google Drive files

- MyDrive/AgroVision/dataset/plant_village_dataset.zip — audited dataset archive.
- MyDrive/AgroVision/splits/plantvillage_seed42.csv — copy of AgroVision_AI/ml/splits/plantvillage_seed42.csv.
- MyDrive/AgroVision/config/class_labels.json — copy of AgroVision_AI/backend/class_labels.json.
- MyDrive/AgroVision/models/baseline_mobilenetv2/ — output folder; the notebook creates it.

Edit the paths in the next cell if your Drive layout differs. Keep the ZIP and manifest unchanged.

In [ ]:
# Run settings: edit before training.
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/AgroVision")
DATASET_ZIP = DRIVE_ROOT / "dataset/plant_village_dataset.zip"
SPLIT_CSV = DRIVE_ROOT / "splits/plantvillage_seed42.csv"
CLASS_LABELS_JSON = DRIVE_ROOT / "config/class_labels.json"
OUTPUT_DIR = DRIVE_ROOT / "models/baseline_mobilenetv2"
EXTRACT_DIR = Path("/content/agrovision_plantvillage/extracted")

# Phase 2 archive SHA-256. Set None only for a separately verified archive.
EXPECTED_ARCHIVE_SHA256 = "0123dd0ea9b26edeb5565e04b29c5cc22e9c6f4f829fba28894bbbb459ecffbc"

IMG_SIZE = 224
BATCH_SIZE = 16
HEAD_EPOCHS = 8
FINETUNE_EPOCHS = 15
HEAD_LEARNING_RATE = 1e-3
FINETUNE_LEARNING_RATE = 1e-5
DROPOUT = 0.5
SEED = 42
SHUFFLE_BUFFER = 10000
FINE_TUNE_FRACTION = 0.30
PATIENCE = 4

In [ ]:
import hashlib, json, os, platform, random, shutil, stat, time, traceback, zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd
import tensorflow as tf
from google.colab import drive
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
if not gpus:
    raise RuntimeError(
        "No TensorFlow GPU was detected. In Colab choose Runtime > Change runtime type > GPU, "
        "reconnect, and run again. This notebook is designed for Colab GPU training."
    )
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass
    details = tf.config.experimental.get_device_details(gpu)
    print("GPU:", details.get("device_name", gpu.name))

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
determinism_enabled = True
try:
    tf.config.experimental.enable_op_determinism()
except (AttributeError, RuntimeError) as exc:
    determinism_enabled = False
    print("Could not enable deterministic TensorFlow ops:", str(exc))

print("Python:", platform.python_version())
drive.mount("/content/drive")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Dataset extraction and integrity checks

The archive is read only and extracted under /content, not into Drive or the project. Checks compare ZIP image inventory, class folders, and every manifest path against the authoritative CSV. Label IDs and duplicate-group partition isolation are checked before training.

In [ ]:
def sha256_file(path, chunk=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()


def safe_extract_zip(zip_path, destination):
    """Extract while rejecting absolute paths, traversal, and symlinks."""
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    suffixes = {".jpg", ".jpeg", ".png", ".bmp"}
    image_count = 0
    with zipfile.ZipFile(zip_path) as archive:
        for info in archive.infolist():
            member = PurePosixPath(info.filename.replace("\\", "/"))
            if member.is_absolute() or not member.parts or any(p in {"..", ""} for p in member.parts):
                raise ValueError(f"Unsafe ZIP path: {info.filename!r}")
            target = (destination / Path(*member.parts)).resolve()
            try:
                target.relative_to(root)
            except ValueError as exc:
                raise ValueError(f"ZIP path escapes extraction directory: {info.filename!r}") from exc
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            if ((info.external_attr >> 16) & 0o170000) == stat.S_IFLNK:
                raise ValueError(f"ZIP symlink is not allowed: {info.filename!r}")
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as source, open(target, "wb") as output:
                shutil.copyfileobj(source, output, length=1024 * 1024)
            if Path(member.name).suffix.lower() in suffixes:
                image_count += 1
    return image_count


for required in (DATASET_ZIP, SPLIT_CSV, CLASS_LABELS_JSON):
    if not required.is_file():
        raise FileNotFoundError(f"Missing required Drive file: {required}. See the Drive layout above or edit the paths.")

archive_sha256 = sha256_file(DATASET_ZIP)
print("Dataset ZIP:", DATASET_ZIP)
print("Dataset ZIP SHA-256:", archive_sha256)
if EXPECTED_ARCHIVE_SHA256 and archive_sha256.lower() != EXPECTED_ARCHIVE_SHA256.lower():
    raise ValueError("ZIP hash differs from the Phase 2 audited archive. Verify the copy or update the expected hash only after independently verifying another archive.")
manifest_sha256 = sha256_file(SPLIT_CSV)
archive_image_count = safe_extract_zip(DATASET_ZIP, EXTRACT_DIR)
print("Image files listed in ZIP:", archive_image_count)

In [ ]:
manifest = pd.read_csv(SPLIT_CSV, dtype={"archive_member_path": str, "archive_class_name": str, "class_label": str})
required_columns = {
    "archive_member_path", "archive_class_name", "class_id", "class_label", "split",
    "duplicate_group_id", "duplicate_group_size", "duplicate_group_reason",
}
missing_columns = required_columns - set(manifest.columns)
if missing_columns:
    raise ValueError(f"Split manifest missing columns: {sorted(missing_columns)}")
manifest["class_id"] = pd.to_numeric(manifest["class_id"], errors="raise").astype(int)
manifest["split"] = manifest["split"].astype(str)
allowed = {"train", "validation", "test"}
if set(manifest["split"]) != allowed or manifest["archive_member_path"].duplicated().any():
    raise ValueError("Manifest partitions or image paths are invalid.")

with open(CLASS_LABELS_JSON, encoding="utf-8") as stream:
    labels_by_id = {int(k): str(v) for k, v in json.load(stream).items()}
class_ids = sorted(labels_by_id)
if class_ids != list(range(38)):
    raise ValueError(f"Expected class IDs 0-37; found {class_ids}")
class_names = [labels_by_id[i] for i in class_ids]
manifest_classes = manifest[["class_id", "class_label"]].drop_duplicates()
if set(manifest_classes["class_id"]) != set(class_ids) or manifest_classes["class_id"].nunique() != 38:
    raise ValueError("Manifest does not contain exactly the 38 expected class IDs.")
for row in manifest_classes.itertuples(index=False):
    if labels_by_id[int(row.class_id)] != str(row.class_label):
        raise ValueError(f"Manifest label mismatch at class ID {row.class_id}: {row.class_label!r}")

expected_counts = {"train": 37991, "validation": 8158, "test": 8154}
partition_counts = manifest["split"].value_counts().to_dict()
print("Manifest rows:", len(manifest), "| partition counts:", partition_counts)
if partition_counts != expected_counts:
    raise ValueError(f"Partition counts differ from audited seed-42 split: {partition_counts}")
if archive_image_count != len(manifest):
    raise ValueError(f"ZIP has {archive_image_count} image entries but manifest has {len(manifest)} rows.")

archive_paths = manifest["archive_member_path"].map(PurePosixPath)
if any(p.is_absolute() or len(p.parts) != 2 or ".." in p.parts for p in archive_paths):
    raise ValueError("Manifest paths must be safe class-folder/image paths.")
if any(p.parts[0] != row.archive_class_name for p, row in zip(archive_paths, manifest.itertuples(index=False))):
    raise ValueError("Manifest folder name differs from archive_class_name.")
manifest["image_path"] = [str((EXTRACT_DIR / Path(*p.parts)).resolve()) for p in archive_paths]
missing = [p for p in manifest["image_path"] if not Path(p).is_file()]
if missing:
    raise FileNotFoundError(f"{len(missing)} manifest images are missing; example: {missing[0]}")

suffixes = {".jpg", ".jpeg", ".png", ".bmp"}
extracted_images = [p for p in EXTRACT_DIR.rglob("*") if p.is_file() and p.suffix.lower() in suffixes]
folder_classes = {p.relative_to(EXTRACT_DIR).parts[0] for p in extracted_images}
if len(extracted_images) != len(manifest) or folder_classes != set(manifest["archive_class_name"]) or len(folder_classes) != 38:
    raise ValueError("Extracted image count or class folders do not match the manifest.")
if (manifest.groupby("duplicate_group_id")["split"].nunique() > 1).any():
    raise ValueError("A duplicate group crosses train/validation/test partitions.")
if any(manifest.loc[manifest.split == part, "class_id"].nunique() != 38 for part in allowed):
    raise ValueError("Every partition must contain all 38 classes.")

class_distribution = manifest.groupby(["split", "class_id", "class_label"]).size().rename("image_count").reset_index()
train_manifest = manifest.loc[manifest.split == "train"].reset_index(drop=True)
validation_manifest = manifest.loc[manifest.split == "validation"].reset_index(drop=True)
test_count_verified = int((manifest.split == "test").sum())
print("Verified images:", len(extracted_images), "| classes:", len(folder_classes), "| label/folder checks passed.")
print("Duplicate-group leakage check passed.")

## TensorFlow pipeline and class weights

Images are decoded as RGB and resized to 224×224. The pipeline yields float32 pixels in the 0–255 range. A rescaling layer inside the saved model implements MobileNetV2 preprocessing, x / 127.5 - 1. Augmentation is active only with training=True. Validation/test are never shuffled or augmented. No dataset cache is enabled, avoiding a large memory or storage copy.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(frame, training=False):
    paths = frame["image_path"].astype(str).to_numpy()
    labels = frame["class_id"].astype(np.int32).to_numpy()
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        dataset = dataset.shuffle(min(len(frame), SHUFFLE_BUFFER), seed=SEED, reshuffle_each_iteration=True)

    def decode_resize(path, label):
        image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
        image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE], method="bilinear")
        return tf.cast(image, tf.float32), label  # RGB values remain in 0..255

    dataset = dataset.map(decode_resize, num_parallel_calls=AUTOTUNE, deterministic=True)
    return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_manifest, training=True)
validation_ds = make_dataset(validation_manifest)
train_counts = train_manifest["class_id"].value_counts().reindex(class_ids, fill_value=0).sort_index()
weights = compute_class_weight("balanced", classes=np.asarray(class_ids), y=train_manifest["class_id"].to_numpy())
class_weights = {int(i): float(w) for i, w in zip(class_ids, weights)}
class_distribution.to_csv(OUTPUT_DIR / "class_distribution.csv", index=False)
print("Actual train class distribution:")
print(pd.DataFrame({"class_id": class_ids, "class_label": class_names, "train_images": train_counts.values}).to_string(index=False))
print("Balanced train-only class weights:")
print(json.dumps(class_weights, indent=2))

## New MobileNetV2 model

The head is GlobalAveragePooling2D → BatchNormalization → Dense(256, ReLU) → Dropout(0.5) → Dense(38, softmax). Stage 1 freezes the ImageNet backbone. The model accepts RGB pixels in 0–255, performs moderate augmentation during training, then maps inputs to the MobileNetV2 range. No weights are loaded from the historical application model.

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal", seed=SEED),
    layers.RandomRotation(0.06, fill_mode="reflect", seed=SEED + 1),
    layers.RandomTranslation(0.06, 0.06, fill_mode="reflect", seed=SEED + 2),
    layers.RandomZoom(0.08, fill_mode="reflect", seed=SEED + 3),
    layers.RandomContrast(0.10, seed=SEED + 4),
], name="train_only_augmentation")

inputs = layers.Input((IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32, name="rgb_pixels_0_255")
x = layers.Rescaling(1.0 / 255.0, name="pixels_0_to_1")(inputs)
x = augmentation(x)
x = layers.Rescaling(2.0, offset=-1.0, name="mobilenetv2_preprocessing_equivalent_to_x_over_127_5_minus_1")(x)
backbone = MobileNetV2(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3), name="mobilenetv2_backbone")
backbone.trainable = False
x = backbone(x, training=False)
x = layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
x = layers.BatchNormalization(name="head_batch_normalization")(x)
x = layers.Dense(256, activation="relu", name="head_dense_256")(x)
x = layers.Dropout(DROPOUT, seed=SEED, name="head_dropout")(x)
outputs = layers.Dense(38, activation="softmax", name="class_probabilities")(x)
model = models.Model(inputs, outputs, name="agrovision_mobilenetv2_baseline")


def parameter_counts(m):
    total = int(sum(np.prod(w.shape) for w in m.weights))
    trainable = int(sum(np.prod(w.shape) for w in m.trainable_weights))
    return {"total": total, "trainable": trainable, "non_trainable": total - trainable}


def save_model_summary(m, path):
    lines = []
    m.summary(print_fn=lines.append)
    counts = parameter_counts(m)
    lines += ["", f"Input shape: {m.input_shape}", f"Output shape: {m.output_shape}",
              f"Total parameters: {counts['total']:,}", f"Trainable parameters: {counts['trainable']:,}",
              f"Non-trainable parameters: {counts['non_trainable']:,}"]
    Path(path).write_text("\n".join(lines) + "\n", encoding="utf-8")
    print("\n".join(lines[-6:]))
    return counts

model.compile(optimizer=tf.keras.optimizers.Adam(HEAD_LEARNING_RATE),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
stage1_parameter_counts = save_model_summary(model, OUTPUT_DIR / "model_summary.txt")

## Two-stage transfer learning

Stage 1 trains the head while the backbone is frozen. Stage 2 starts from Stage 1's best validation checkpoint, unfreezes the final configured fraction of MobileNetV2 layers, keeps BatchNormalization frozen, and recompiles with a substantially smaller Adam learning rate. Checkpoint selection, early stopping, and learning-rate reduction use validation loss only.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

stage1_path = OUTPUT_DIR / "stage1_best.keras"
best_model_path = OUTPUT_DIR / "best_model.keras"

def callbacks_for(path):
    return [
        ModelCheckpoint(str(path), monitor="val_loss", mode="min", save_best_only=True, verbose=1),
        EarlyStopping(monitor="val_loss", mode="min", patience=PATIENCE, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", mode="min", factor=0.3, patience=2, min_lr=1e-7, verbose=1),
    ]

def history_record(history, stage, seconds):
    return {"stage": stage, "elapsed_seconds": float(seconds),
            "history": {k: [float(v) for v in values] for k, values in history.history.items()}}

run_started_utc = datetime.now(timezone.utc).isoformat()
stage_histories = []
training_started = time.perf_counter()
try:
    print("STAGE 1 — frozen backbone")
    started = time.perf_counter()
    history1 = model.fit(train_ds, validation_data=validation_ds, epochs=HEAD_EPOCHS,
                         class_weight=class_weights, callbacks=callbacks_for(stage1_path))
    stage_histories.append(history_record(history1, "head_training", time.perf_counter() - started))

    model = tf.keras.models.load_model(stage1_path)
    backbone = model.get_layer("mobilenetv2_backbone")
    backbone.trainable = True
    fine_tune_start = int(len(backbone.layers) * (1.0 - FINE_TUNE_FRACTION))
    for i, layer in enumerate(backbone.layers):
        layer.trainable = i >= fine_tune_start and not isinstance(layer, tf.keras.layers.BatchNormalization)
    trainable_backbone_layers = sum(layer.trainable for layer in backbone.layers)
    if not trainable_backbone_layers:
        raise RuntimeError("Fine-tuning selected no trainable backbone layers.")
    print(f"Unfrozen backbone layers: {trainable_backbone_layers}/{len(backbone.layers)}; BatchNormalization frozen.")

    model.compile(optimizer=tf.keras.optimizers.Adam(FINETUNE_LEARNING_RATE),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    print("STAGE 2 — fine-tuning")
    started = time.perf_counter()
    history2 = model.fit(train_ds, validation_data=validation_ds, epochs=FINETUNE_EPOCHS,
                         class_weight=class_weights, callbacks=callbacks_for(best_model_path))
    stage_histories.append(history_record(history2, "fine_tuning", time.perf_counter() - started))
    training_seconds = time.perf_counter() - training_started

    # Restore the best validation checkpoint; no fit calls occur after this point.
    model = tf.keras.models.load_model(best_model_path)
    final_parameter_counts = save_model_summary(model, OUTPUT_DIR / "model_summary.txt")
    model.save(best_model_path)
    h5_model_path = OUTPUT_DIR / "best_model.h5"
    h5_saved = False
    try:
        model.save(h5_model_path)
        h5_saved = True
        print("Saved optional HDF5 model:", h5_model_path)
    except Exception as exc:
        print("Optional HDF5 export unavailable:", repr(exc))
        if h5_model_path.exists():
            h5_model_path.unlink()
except Exception:
    (OUTPUT_DIR / "training_error.txt").write_text(traceback.format_exc(), encoding="utf-8")
    raise

training_history = {"run_started_utc": run_started_utc,
                    "training_elapsed_seconds": float(training_seconds), "stages": stage_histories}
(OUTPUT_DIR / "training_history.json").write_text(json.dumps(training_history, indent=2), encoding="utf-8")
print(f"Actual total training time: {training_seconds / 60:.2f} minutes")

In [ ]:
gpu_names = [tf.config.experimental.get_device_details(g).get("device_name", g.name) for g in gpus]
config = {
    "run_started_utc": run_started_utc, "python_version": platform.python_version(),
    "tensorflow_version": tf.__version__, "gpu_names": gpu_names,
    "deterministic_ops_enabled": determinism_enabled, "seed": SEED,
    "dataset_zip_path": str(DATASET_ZIP), "dataset_zip_sha256": archive_sha256,
    "dataset_source_reference": "https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset",
    "dataset_source_provenance": "Local archive Kaggle origin/version and license were not verified; confirm before publication.",
    "split_manifest_path": str(SPLIT_CSV), "split_manifest_sha256": manifest_sha256,
    "verified_image_count": int(len(manifest)), "verified_class_count": len(class_ids),
    "partition_counts": {k: int(v) for k, v in partition_counts.items()},
    "class_distribution": class_distribution.to_dict(orient="records"),
    "class_weight_strategy": "sklearn compute_class_weight('balanced') on train split only",
    "class_weights": {str(k): v for k, v in class_weights.items()},
    "IMG_SIZE": IMG_SIZE, "BATCH_SIZE": BATCH_SIZE, "HEAD_EPOCHS": HEAD_EPOCHS,
    "FINETUNE_EPOCHS": FINETUNE_EPOCHS, "HEAD_LEARNING_RATE": HEAD_LEARNING_RATE,
    "FINETUNE_LEARNING_RATE": FINETUNE_LEARNING_RATE, "DROPOUT": DROPOUT,
    "FINE_TUNE_FRACTION": FINE_TUNE_FRACTION, "PATIENCE": PATIENCE,
    "shuffle_buffer": SHUFFLE_BUFFER, "parameter_counts_after_fine_tuning": final_parameter_counts,
    "training_elapsed_seconds": float(training_seconds), "h5_export_saved": h5_saved,
    "augmentation": {"horizontal_flip": True, "rotation": 0.06, "translation": 0.06,
                     "zoom": 0.08, "contrast": 0.10, "training_only": True},
}
(OUTPUT_DIR / "training_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
shutil.copy2(CLASS_LABELS_JSON, OUTPUT_DIR / "class_labels.json")
shutil.copy2(SPLIT_CSV, OUTPUT_DIR / "split_manifest.csv")
preprocessing_config = {
    "input_shape": [IMG_SIZE, IMG_SIZE, 3], "color_order": "RGB", "dtype": "float32",
    "input_range": [0.0, 255.0],
    "layers": ["divide by 255", "training-only augmentation", "multiply by 2 and subtract 1"],
    "equivalent_formula": "x / 127.5 - 1.0", "validation_test_augmentation": False,
}
(OUTPUT_DIR / "preprocessing_config.json").write_text(json.dumps(preprocessing_config, indent=2), encoding="utf-8")

## Training curves and validation evaluation

The selected checkpoint is best_model.keras, chosen only by validation loss. The next cell plots actual per-epoch history and calculates validation accuracy, precision, recall, macro/weighted F1, top-3 accuracy, per-class metrics, and the confusion matrix. Results are explicitly labeled VALIDATION.

In [ ]:
import matplotlib.pyplot as plt

curves = {"accuracy": [], "val_accuracy": [], "loss": [], "val_loss": []}
for stage in stage_histories:
    for key in curves:
        curves[key].extend(stage["history"].get(key, []))
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(curves["accuracy"], label="train")
axes[0].plot(curves["val_accuracy"], label="validation")
axes[0].set(title="Accuracy by epoch", xlabel="Epoch (Stage 1 then Stage 2)", ylabel="Accuracy")
axes[0].legend()
axes[1].plot(curves["loss"], label="train")
axes[1].plot(curves["val_loss"], label="validation")
axes[1].set(title="Loss by epoch", xlabel="Epoch (Stage 1 then Stage 2)", ylabel="Loss")
axes[1].legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "training_validation_curves.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

# Inference-only object. Do not fit or tune after this point.
best_model = tf.keras.models.load_model(best_model_path, compile=False)

def calculate_metrics(keras_model, dataset, true_ids, prefix):
    probabilities = keras_model.predict(dataset, verbose=1)
    predicted_ids = np.argmax(probabilities, axis=1)
    top3_ids = np.argsort(probabilities, axis=1)[:, -3:]
    report = classification_report(true_ids, predicted_ids, labels=class_ids,
                                   target_names=class_names, output_dict=True, zero_division=0)
    matrix = confusion_matrix(true_ids, predicted_ids, labels=class_ids)
    per_class = [{
        "class_label": name, "precision": float(report[name]["precision"]),
        "recall": float(report[name]["recall"]), "f1_score": float(report[name]["f1-score"]),
        "support": int(report[name]["support"]),
    } for name in class_names]
    metrics = {
        "result_split": prefix, "sample_count": int(len(true_ids)),
        "accuracy": float(accuracy_score(true_ids, predicted_ids)),
        "precision_macro": float(report["macro avg"]["precision"]),
        "recall_macro": float(report["macro avg"]["recall"]),
        "f1_macro": float(report["macro avg"]["f1-score"]),
        "precision_weighted": float(report["weighted avg"]["precision"]),
        "recall_weighted": float(report["weighted avg"]["recall"]),
        "f1_weighted": float(report["weighted avg"]["f1-score"]),
        "top3_accuracy": float(np.mean([int(y) in row for y, row in zip(true_ids, top3_ids)])),
        "per_class_metrics": per_class, "confusion_matrix": matrix.tolist(),
        "class_label_order": class_names,
    }
    (OUTPUT_DIR / f"{prefix.lower()}_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    pd.DataFrame(per_class).to_csv(OUTPUT_DIR / f"{prefix.lower()}_per_class_metrics.csv", index=False)
    pd.DataFrame(matrix, index=class_names, columns=class_names).to_csv(
        OUTPUT_DIR / f"{prefix.lower()}_confusion_matrix.csv", index_label="actual_class")
    print(prefix, "results:")
    print(json.dumps({k: v for k, v in metrics.items()
                      if k not in {"per_class_metrics", "confusion_matrix", "class_label_order"}}, indent=2))
    return metrics

validation_ids = validation_manifest["class_id"].to_numpy(dtype=np.int32)
validation_metrics = calculate_metrics(best_model, validation_ds, validation_ids, "VALIDATION")

## Efficiency measurements

Parameter counts and model size come from the saved baseline. Batch-one latency uses validation images, warm-up calls, and synchronized predictions. CPU and GPU times are labeled; actual latency depends on Colab hardware and load.

In [ ]:
def measure_batch1_latency(keras_model, device, image_batch, iterations=100, warmup=10):
    samples = []
    with tf.device(device):
        one = tf.identity(image_batch[:1])
        for _ in range(warmup):
            keras_model(one, training=False).numpy()
        for _ in range(iterations):
            started = time.perf_counter()
            keras_model(one, training=False).numpy()
            samples.append((time.perf_counter() - started) * 1000.0)
    return {"device": device, "batch_size": 1, "iterations": iterations,
            "mean_ms_per_image": float(np.mean(samples)),
            "median_ms_per_image": float(np.median(samples)),
            "p95_ms_per_image": float(np.percentile(samples, 95))}

model_counts = parameter_counts(model)
model_size_bytes = int(best_model_path.stat().st_size)
sample_batch, _ = next(iter(validation_ds))
gpu_name = tf.config.experimental.get_device_details(gpus[0]).get("device_name", gpus[0].name)
gpu_latency = measure_batch1_latency(best_model, "/GPU:0", sample_batch)
with tf.device("/CPU:0"):
    cpu_model = tf.keras.models.load_model(best_model_path, compile=False)
cpu_latency = measure_batch1_latency(cpu_model, "/CPU:0", sample_batch)
efficiency = {
    "total_parameters": model_counts["total"],
    "trainable_parameters_after_fine_tuning": model_counts["trainable"],
    "non_trainable_parameters_after_fine_tuning": model_counts["non_trainable"],
    "keras_model_file": str(best_model_path), "keras_model_size_bytes": model_size_bytes,
    "keras_model_size_mib": float(model_size_bytes / (1024 ** 2)), "gpu_name": gpu_name,
    "gpu_batch1_latency": gpu_latency, "cpu_batch1_latency": cpu_latency,
}
if h5_saved and h5_model_path.exists():
    efficiency["h5_model_size_bytes"] = int(h5_model_path.stat().st_size)
(OUTPUT_DIR / "efficiency_metrics.json").write_text(json.dumps(efficiency, indent=2), encoding="utf-8")
print(json.dumps(efficiency, indent=2))

## Final TEST evaluation — once, after model selection

Training and validation-based checkpoint selection are complete. Do not fit or tune after this point. This final section creates the held-out test dataset for the first time, evaluates once, and saves the metrics as TEST results. Do not use these results to revise the baseline.

In [ ]:
# Deliberately access test rows and build the test pipeline only in this final evaluation cell.
test_manifest = manifest.loc[manifest.split == "test"].reset_index(drop=True)
if len(test_manifest) != test_count_verified or len(test_manifest) != expected_counts["test"]:
    raise RuntimeError("Verified test count changed before final evaluation.")
test_ds = make_dataset(test_manifest, training=False)
test_ids = test_manifest["class_id"].to_numpy(dtype=np.int32)
test_metrics = calculate_metrics(best_model, test_ds, test_ids, "TEST")

baseline_report = {
    "report_generated_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": {
        "source_reference": config["dataset_source_reference"],
        "source_provenance_limitation": config["dataset_source_provenance"],
        "archive_path": str(DATASET_ZIP), "archive_sha256": archive_sha256,
        "verified_image_count": int(len(manifest)), "verified_class_count": len(class_ids),
        "class_distribution": class_distribution.to_dict(orient="records"),
        "train_count": int(len(train_manifest)), "validation_count": int(len(validation_manifest)),
        "test_count": int(len(test_manifest)), "split_manifest_path": str(SPLIT_CSV),
        "split_manifest_sha256": manifest_sha256,
        "duplicate_group_leakage_check": "passed; duplicate groups confined to one partition",
    },
    "model": {
        "architecture": "ImageNet MobileNetV2, GlobalAveragePooling2D, BatchNormalization, Dense(256,relu), Dropout(0.5), Dense(38,softmax)",
        "preprocessing": preprocessing_config, "parameter_counts": model_counts,
        "input_shape": list(best_model.input_shape), "output_shape": list(best_model.output_shape),
    },
    "training": {
        "strategy": "frozen backbone head stage, then upper-backbone fine-tuning with BatchNormalization frozen",
        "seed": SEED, "batch_size": BATCH_SIZE, "head_epoch_limit": HEAD_EPOCHS,
        "fine_tune_epoch_limit": FINETUNE_EPOCHS, "head_learning_rate": HEAD_LEARNING_RATE,
        "fine_tune_learning_rate": FINETUNE_LEARNING_RATE, "augmentation": config["augmentation"],
        "class_weight_strategy": config["class_weight_strategy"], "class_weights": config["class_weights"],
        "elapsed_seconds": float(training_seconds), "actual_stage_histories": stage_histories,
    },
    "validation_results": validation_metrics, "test_results": test_metrics, "efficiency": efficiency,
    "environment": {"python_version": platform.python_version(), "tensorflow_version": tf.__version__,
                    "gpu_name": gpu_name, "deterministic_ops_enabled": determinism_enabled},
    "limitations": [
        "PlantVillage images are controlled, mostly isolated leaf images and may not represent field conditions, device variation, backgrounds, lighting, or disease severity.",
        "Local archive Kaggle provenance and license were not verified; confirm source, version, and license before publication.",
        "Seeds and deterministic ops improve repeatability, but hardware, TensorFlow/CUDA versions, and nondeterministic kernels can affect exact results.",
        "Colab GPU availability and runtime limits can interrupt training; actual histories and training_error.txt record execution.",
    ],
}
(OUTPUT_DIR / "baseline_report.json").write_text(json.dumps(baseline_report, indent=2), encoding="utf-8")
print("Saved final report:", OUTPUT_DIR / "baseline_report.json")
print("All reported metric values came from this notebook execution; none are prefilled.")

## Output files

The Drive output folder receives best_model.keras, optional best_model.h5, the Stage 1 checkpoint, training history/configuration, class labels, preprocessing configuration, model summary, class distribution, training/validation plots, validation and test metrics/CSV confusion matrices, efficiency measurements, and baseline_report.json. This notebook contains no fabricated result values.